<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

<br>

# <font color="#76b900">**Notebook 8 [Assessment]:** RAG Evaluation</font>

<br>

Welcome to the last notebook of the course! In the previous notebook, you integrated a vector store solution into a RAG pipeline! In this notebook, you will take that same pipeline and evaluate it using numerical RAG evaluation techniques incorporating LLM-as-a-Judge metrics!

<br>

### **Learning Objectives:**

- Learn how to integrate the techniques from prior notebooks to numerically approximate the goodness of your RAG pipeline.

- **Final Exercice**: ***By working through this notebook in the Course Environment,* you will be able to submit the coding component of the course!**

<br>

### **Questions To Think About:**

- As you go along, remember what our metrics actually represent. Should our pipeline pass these objectives? Is our judge LLM sufficient for evaluating the pipeline? Does a particular metric even matter for our use case?
- If we left the vectorstore-as-a-memory component in our chain, do you think it would still pass the evaluation? Additionally, is the evaluation useful for assessing vectorstore-as-a-memory performance? 

<br>

### **Environment Setup:**

In [1]:
# %pip install -q langchain langchain-nvidia-ai-endpoints gradio rich
# %pip install -q arxiv pymupdf faiss-cpu ragas

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
norm_style = Style(bold=True)
pprint = partial(console.print, style=base_style)
pprint2 = partial(console.print, style=norm_style)

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

# NVIDIAEmbeddings.get_available_models()
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")

# ChatNVIDIA.get_available_models(base_url="http://llm_client:9000/v1")
instruct_llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

----

<br>

## **Part 1:** Pre-Release Evaluation

In our previous notebook, we successfully combined several concepts to create a document chatbot with the aim of responsive and informative interactions. However, the diversity of user interactions necessitates comprehensive testing to truly understand the chatbot's performance. Thorough testing in varied scenarios is crucial to ensure that the system is not only robust and versatile but also aligns with user and provider expectations.

After defining your chatbot's roles and implementing the necessary features, evaluating it becomes a multi-stage process:

- **Typical Use Inspection:** Start by testing scenarios most relevant to your use case. See if your chatbot can reliably navigate discussions with limited human intervention.

    - Additionally, identify limitations or compartments that should be redirected to a human for inspection/supervision (i.e., human swap-in to confirm transactions or perform sensitive navigation) and implement those options.

- **Edge Case Inspection:** Explore the boundaries of typical use, identifying how the chatbot handles less common but plausible scenarios.

    - Before any public release, assess critical boundary conditions that could pose liability risks, such as the potential generation of inappropriate content.

    - Implement well-tested guardrails on all outputs (and possibly inputs) to limit undesired interactions and redirect users into predictable conversation flows.

- **Progressive Rollout:** Rolling out your model to a limited audience (first internal, then [A/B](https://en.wikipedia.org/wiki/A/B_testing)) and implement analytics features like usage analytics dashboards and feedback avenues (flag/like/dislike/etc).

Of these three steps, the first two can be done by a small team or an individual and should be iterated on as part of the development process. Unfortunately, this needs to be done frequently and can be prone to human error. **Luckily for us, LLMs can be used to help out with LLM-as-a-Judge formulations!**

*(Yeah, this probably isn't surprising by now. LLMs being strong is why this course is here...).*

----

<br>

## **Part 2:** LLM-as-a-Judge Formulation

In the realm of conversational AI, using LLMs as evaluators or 'judges' has emerged as a useful approach for configurable automatic testing of natural language task performance:

- An LLM can simulate a range of interaction scenarios and generate synthetic data, allowing an evaluation developer to generate targeted inputs to eliciting a range of behaviors from your chatbot.

- The chatbot's correspondence/retrieval on the synthetic data can be evaluated or parsed by an LLM and a consistent output format such as "Pass"/"Fail", similarity, or extraction can be enforced.

- Many such results can be aggregated and a metric can be derived which explains something like "% of passing evaluations", "average number of relevant details from the sources", "average cosine similarity", etc.

This idea of using LLMs to test out and quantify chatbot quality, known as [**"LLM-as-a-Judge,"**](https://arxiv.org/abs/2306.05685) allows for easy test specifications that align closely with human judgment and can be fine-tuned and replicated at scale.

**There are several popular frameworks for off-the-shelf judge formulations including:**
- [**RAGAs (short for RAG Assessment)**](https://docs.ragas.io/en/stable/), which offers a suite of great starting points for your own evaluation efforts.
- [**LangChain Evaluators**](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/), which are similar first-party options with many implicitly-constructible agents.

Instead of using the chains as-is, we will instead expand on the ideas and evaluate our system with a more custom solution.

----

<br>

## **Part 3: [Assessment Prep]** Pairwise Evaluator

The following exercise will flesh out a custom implementation of a simplified [LangChain Pairwise String Evaluator](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/comparison/pairwise_string/). 

**To prepare for our RAG chain evaluation, we will need to:**

- Pull in our document index (the one we saved in the previous notebook).
- Recreate our RAG pipeline of choice.

**We will specifically be implementing a judge formulation with the following steps:**

- Sample the RAG agent document pool to find two document chunks.
- Use those two document chunks to generate a synthetic "baseline" question-answer pair.
- Use the RAG agent to generate its own answer.
- Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."

**The chain should be a simple but powerful process that tests for the following objective:**

> ***Does my RAG chain outperform a narrow chatbot with limited document access.***





**This will be the system used for the final evaluation!** To see how this system is integrated into the autograder, please check out the implementation in [`frontend/server_app.py`](frontend/server_app.py).

<br>

### **Task 1:** Pull In Your Document Retrieval Index

For this exercise, you will pull in the `docstore_index` file you created as part of your earlier notebook. The following cell should be able to load in the store as-is.

In [2]:
## Make sure you have docstore_index.tgz in your working directory
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS

# embedder = NVIDIAEmbeddings(model="nvidia/embed-qa-4", truncate="END")

!tar xzvf docstore_index.tgz
docstore = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
docs = list(docstore.docstore._dict.values())

def format_chunk(doc):
    return (
        f"Paper: {doc.metadata.get('Title', 'unknown')}"
        f"\n\nSummary: {doc.metadata.get('Summary', 'unknown')}"
        f"\n\nPage Body: {doc.page_content}"
    )

## This printout just confirms that your store has been retrieved
pprint(f"Constructed aggregate docstore with {len(docstore.docstore._dict)} chunks")
pprint(f"Sample Chunk:")
print(format_chunk(docs[len(docs)//2]))

docstore_index/
docstore_index/index.pkl
docstore_index/index.faiss


Constructed aggregate docstore with 74 chunks

Sample Chunk:

Paper: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks

Summary: Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when fine-tuned on
downstream NLP tasks. However, their ability to access and precisely manipulate
knowledge is still limited, and hence on knowledge-intensive tasks, their
performance lags behind task-specific architectures. Additionally, providing
provenance for their decisions and updating their world knowledge remain open
research problems. Pre-trained models with a differentiable access mechanism to
explicit non-parametric memory can overcome this issue, but have so far been
only investigated for extractive downstream tasks. We explore a general-purpose
fine-tuning recipe for retrieval-augmented generation (RAG) -- models which
combine pre-trained parametric and non-parametric memory for language
generation. We introduce RAG models where the parametric memory is a
pre

<br>

### **Task 2: [Exercise]** Pull In Your RAG Chain

Now that we have our index, we can recreate the RAG agent from the previous notebook! 

**Key Modifications:**
- To keep things simple, feel free to disregard the vectorstore-as-a-memory component. Incorporating it will require some more overhead and will make the exercise a bit more complicated.

In [3]:
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableLambda, RunnableBranch
# from langchain_core.runnables.passthrough import RunnableAssign
# from langchain.document_transformers import LongContextReorder

# from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

# from functools import partial
# from operator import itemgetter

# import gradio as gr

# #####################################################################

# # NVIDIAEmbeddings.get_available_models()
# embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")

# # ChatNVIDIA.get_available_models()
# instruct_llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")
# llm = instruct_llm | StrOutputParser()

# #####################################################################

# def docs2str(docs, title="Document"):
#     """Useful utility for making chunks into context string. Optional, but useful"""
#     out_str = ""
#     for doc in docs:
#         doc_name = getattr(doc, 'metadata', {}).get('Title', title)
#         if doc_name: out_str += f"[Quote from {doc_name}] "
#         out_str += getattr(doc, 'page_content', str(doc)) + "\n"
#     return out_str

# chat_prompt = ChatPromptTemplate.from_template(
#     "You are a document chatbot. Help the user as they ask questions about documents."
#     " User messaged just asked you a question: {input}\n\n"
#     " The following information may be useful for your response: "
#     " Document Retrieval:\n{context}\n\n"
#     " (Answer only from retrieval. Only cite sources that are used. Make your response conversational)"
#     "\n\nUser Question: {input}"
# )

# def output_puller(inputs):
#     """"Output generator. Useful if your chain returns a dictionary with key 'output'"""
#     if isinstance(inputs, dict):
#         inputs = [inputs]
#     for token in inputs:
#         if token.get('output'):
#             yield token.get('output')

# #####################################################################
# ## TODO: Pull in your desired RAG Chain. Memory not necessary

# ## Chain 1 Specs: "Hello World" -> retrieval_chain 
# ##   -> {'input': <str>, 'context' : <str>}
# long_reorder = RunnableLambda(LongContextReorder().transform_documents)  ## GIVEN
# context_getter = RunnableLambda(lambda x: x)  ## TODO
# retrieval_chain = {'input' : (lambda x: x)} | RunnableAssign({'context' : context_getter})

# ## Chain 2 Specs: retrieval_chain -> generator_chain 
# ##   -> {"output" : <str>, ...} -> output_puller
# generator_chain = RunnableLambda(lambda x: x)  ## TODO
# generator_chain = {'output' : generator_chain} | RunnableLambda(output_puller)  ## GIVEN

# ## END TODO
# #####################################################################

# rag_chain = retrieval_chain | generator_chain

# # pprint(rag_chain.invoke("Tell me something interesting!"))
# for token in rag_chain.stream("Tell me something interesting!"):
#     print(token, end="")



from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableAssign
from langchain.document_transformers import LongContextReorder
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import ArxivLoader
from functools import partial
import gradio as gr

#####################################################################
# 🔹 Load embeddings and LLM
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")
instruct_llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")
llm = instruct_llm | StrOutputParser()

#####################################################################
# 🔹 Load and prepare documents
loader = ArxivLoader(query="Retrieval Augmented Generation")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = splitter.split_documents(docs)

# Create FAISS vectorstore
docstore = FAISS.from_documents(splits, embedder)

#####################################################################
# 🔹 Utility to make chunks into readable context
def docs2str(docs, title="Document"):
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name:
            out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str.strip()

#####################################################################
# 🔹 Prompt Template
chat_prompt = ChatPromptTemplate.from_messages([("system",
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked you a question: {input}\n\n"
    " The following information may be useful for your response: "
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational)"
), ('user', '{input}')])


#####################################################################
# 🔹 Step 1: Retrieval Chain
long_reorder = RunnableLambda(LongContextReorder().transform_documents)

retrieval_chain = (
    {"input": lambda x: x}
    | RunnableAssign({
        "context": lambda d: docs2str(
            long_reorder.invoke(docstore.similarity_search(d, k=4))
        )
    })
)

#####################################################################
# 🔹 Step 2: Generation Chain
generator_chain = (
    chat_prompt
    | instruct_llm
    | StrOutputParser()
)

#####################################################################
rag_chain = (
    RunnableLambda(lambda q: {
        "input": q,
        "context": docs2str(docstore.similarity_search(q, k=3))
    })
    | chat_prompt
    | RunnableLambda(lambda p: p.to_string())  # Ensures string input to LLM
    | llm
)

#####################################################################
# ✅ Test it once
print("Testing chain...\n")
for token in rag_chain.stream("Explain Retrieval-Augmented Generation in simple terms."):
    print(token, end="")


Testing chain...

Retrieval-Augmented Generation (RAG) - it's a relatively new concept in the NLP (Natural Language Processing) world.

So, basically, RAG combines two technologies: traditional information retrieval and deep learning generation models. The idea is to use a generation model, like a powerful language model, and enhance its abilities by incorporating external knowledge from a massive collection of documents.

Here's how it works: when you want to generate text, the model doesn't start from scratch; instead, it retrieves relevant documents from the collection based on the input prompt. It's like asking a librarian to find books related to the topic you're interested in. The model then uses these retrieved documents as a foundation to generate text that's potentially more accurate, informative, and coherent.

Think of it like a particularly helpful study buddy. You're working on a project, and your buddy not only has access to the same materials you do but also knows where 

<br>

### **Step 3:** Generating Synthetic Question-Answer Pairs

In this section, we can implement the first few part of our evaluation routine:

- **Sample the RAG agent document pool to find two document chunks.**
- **Use those two document chunks to generate a synthetic "baseline" question-answer pair.**
- Use the RAG agent to generate its own answer.
- Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."

The chain should be a simple but powerful process that tests for the following objective:

> Does my RAG chain outperform a narrow chatbot with limited document access?

In [4]:
import random

num_questions = 3
synth_questions = []
synth_answers = []

simple_prompt = ChatPromptTemplate.from_messages([('system', '{system}'), ('user', 'INPUT: {input}')])

for i in range(num_questions):
    doc1, doc2 = random.sample(docs, 2)
    sys_msg = (
        "Use the documents provided by the user to generate an interesting question-answer pair."
        " Try to use both documents if possible, and rely more on the document bodies than the summary."
        " Use the format:\nQuestion: (good question, 1-3 sentences, detailed)\n\nAnswer: (answer derived from the documents)"
        " DO NOT SAY: \"Here is an interesting question pair\" or similar. FOLLOW FORMAT!"
    )
    usr_msg = (
        f"Document1: {format_chunk(doc1)}\n\n"
        f"Document2: {format_chunk(doc2)}"
    )

    qa_pair = (simple_prompt | llm).invoke({'system': sys_msg, 'input': usr_msg})
    synth_questions += [qa_pair.split('\n\n')[0]]
    synth_answers += [qa_pair.split('\n\n')[1]]
    pprint2(f"QA Pair {i+1}")
    pprint2(synth_questions[-1])
    pprint(synth_answers[-1])
    print()

QA Pair 1

Question: What is the key finding from the study of LLM-based data augmentation for retrieval, concerning the 
diminishing returns observed when scaling up the augmentation model? 

Answer: The study finds that while augmentation improves retrieval models, its benefits do not scale indefinitely; 
beyond a certain point, additional augmentation leads to diminishing returns. This suggests that augmenting all 
documents is not necessary, and a smaller subset can be just as effective.

QA Pair 2

Question: What are the key findings and implications of incorporating retrieval information into retrieval 
augmented generation (R2AG)?

Answer: The key findings and implications of incorporating retrieval information into retrieval augmented 
generation (R2AG) are that it significantly improves the performance of large language models (LLMs) in processing 
multiple documents, especially when those documents include irrelevant information. R2AG enhances the effectiveness
of LLMs in understanding complex dependencies among documents, and its retrieval-aware prompting strategy 
effectively assists LLMs in processing multiple documents. Additionally, R2AG leads to a more comprehensive 
understanding of retrieved documents and improves the generation capabilities of LLMs.

QA Pair 3

Question: What are the key findings from the study that evaluated the effectiveness and scalability of LLM-based 
data augmentation for retrieval?

Answer: The study found that while LLM augmentation is an effective strategy, its benefits do not scale 
indefinitely. The benefits of augmentation diminish beyond a certain threshold, and duplicating the entire dataset 
does not guarantee optimal performance. Additionally, the study demonstrated that smaller augmentation models, such
as those with 8B parameters, can achieve performance comparable to larger models, and retrieving with strong 
pre-training shows diminishing gains from augmentation. Furthermore, the study found that using task-diverse 
augmentations can mitigate performance drops observed in "Fact-Checking" tasks, but the saturation effects persist 
even with task-diverse augmentations. The study also showed that distillation from a larger model to a smaller 
model can bridge the performance gap, making fine-tuning on a small-scale curated subset of high-quality synthetic 
data a viable and cost-effective alternative to large-scale augmentation.

<br>

### **Step 4:** Answer The Synthetic Questions

In this section, we can implement the third part of our evaluation routine:

- Sample the RAG agent document pool to find two document chunks.
- Use those two document chunks to generate a synthetic "baseline" question-answer pair.
- **Use the RAG agent to generate its own answer.**
- Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."

The chain should be a simple but powerful process that tests for the following objective:

> Does my RAG chain outperform a narrow chatbot with limited document access?

In [5]:
rag_answers = []

for i, q in enumerate(synth_questions):
    rag_a = ""
    for token in rag_chain.stream(q):
        if isinstance(token, dict) and "output" in token:
            rag_a += str(token["output"])
        else:
            rag_a += str(token)
    rag_answers.append(rag_a)
    
    pprint2(f"QA Pair {i+1}", q, sep="\n")
    pprint(f"RAG Answer: {rag_a}", "", sep='\n')

QA Pair 1
Question: What is the key finding from the study of LLM-based data augmentation for retrieval, concerning the 
diminishing returns observed when scaling up the augmentation model? 

RAG Answer: So, you're looking for the key finding on diminishing returns. According to the study, it appears that 
there's a point of diminishing returns beyond a certain augmentation threshold. What that means is that while 
LLM-based data augmentation can indeed improve retrieval performance, its advantages don't scale indefinitely. 
augmentation becomes less effective beyond a certain scale, and augmenting the entire corpus is not always 
necessary. In fact, smaller augmentation models, like those with 8B parameters, can achieve performance comparable 
to larger models (70B) with less computational resources!

QA Pair 2
Question: What are the key findings and implications of incorporating retrieval information into retrieval 
augmented generation (R2AG)?

RAG Answer: The key findings of incorporating retrieval information into retrieval augmented generation (R2AG) are 
quite fascinating! 

Based on the paper "R^2AG: Incorporating Retrieval Information into Retrieval Augmented Generation", the results 
show that R2AG is effective, robust, and efficient in capturing retrieval information. The experimental results 
across five datasets demonstrate that R2AG performs better than the original RAG framework.

One of the significant findings is that retrieval information serves as an anchor to aid Large Language Models 
(LLMs) in the generation process. This anchor helps LLMs to understand relationships among documents without 
increasing complexity.

Moreover, the analysis reveals that R2AG increases latency by only 0.8% during inference, making it a suitable 
solution for scenarios with limited resources. This is because R2AG offers the flexibility to fine-tune R2-Former 
alone or both with LLMs, allowing for computational cost savings.

In terms of implications, the study suggests that incorporating retrieval information into RAG can enhance LLMs' 
generation capabilities and relieve information loss. The retrieval-aware prompting strategy designed for R2AG can 
inject retrieval information into the input embeddings, enabling LLMs to better understand the context.

Overall, the findings and implications of R2AG suggest that it is an effective framework for incorporating 
retrieval information into retrieval augmented generation, and its benefits include improved performance, reduced 
complexity, and increased efficiency.

QA Pair 3
Question: What are the key findings from the study that evaluated the effectiveness and scalability of LLM-based 
data augmentation for retrieval?

RAG Answer: The study you're referring to is "Evaluating the Effectiveness and Scalability of LLM-Based Data 
Augmentation for Retrieval." According to the findings, the analysis focused on the impact of augmentation scale 
and model size. The key takeaways are:

1. While augmentation can improve retrieval performance, its advantages do not scale indefinitely. There's a point 
of diminishing returns beyond a certain augmentation threshold, meaning you don't need to augment the entire 
corpus.
2. Smaller augmentation models, like those with 8B parameters, can achieve performance comparable to larger models 
(70B). This suggests a more computationally efficient alternative without sacrificing effectiveness.
3. Models with extensive pre-training show less improvement from augmentation, indicating that augmentation is 
particularly beneficial for models lacking robust pre-training.

These findings highlight the limitations and benefits of LLM-based data augmentation for retrieval, particularly in
relation to model size and pre-training. The study's insights can help inform strategies for optimizing retrieval 
performance in different scenarios.

<br>

### **Step 5:** Implement A Human Preference Metric

In this section, we can implement the fourth part of our evaluation routine:

- Sample the RAG agent document pool to find two document chunks.
- Use those two document chunks to generate a synthetic "baseline" question-answer pair.
- Use the RAG agent to generate its own answer.
- **Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."**

The chain should be a simple but powerful process that tests for the following objective:

> Does my RAG chain outperform a narrow chatbot with limited document access?

In [6]:
## TODO: Adapt this prompt for whichever LLM you're actually interested in using. 
## If it's llama, maybe system message would be good?
eval_prompt = ChatPromptTemplate.from_template("""INSTRUCTION: 
Evaluate the following Question-Answer pair for human preference and consistency.
Assume the first answer is a ground truth answer and has to be correct.
Assume the second answer may or may not be true.
[1] The second answer lies, does not answer the question, or is inferior to the first answer.
[2] The second answer is better than the first and does not introduce any inconsistencies.

Output Format:
[Score] Justification

{qa_trio}

EVALUATION: 
""")

pref_score = []

trio_gen = zip(synth_questions, synth_answers, rag_answers)
for i, (q, a_synth, a_rag) in enumerate(trio_gen):
    pprint2(f"Set {i+1}\n\nQuestion: {q}\n\n")

    qa_trio = f"Question: {q}\n\nAnswer 1 (Ground Truth): {a_synth}\n\n Answer 2 (New Answer): {a_rag}"
    pref_score += [(eval_prompt | llm).invoke({'qa_trio': qa_trio})]
    pprint(f"Synth Answer: {a_synth}\n\n")
    pprint(f"RAG Answer: {a_rag}\n\n")
    pprint2(f"Synth Evaluation: {pref_score[-1]}\n\n")

Set 1

Question: Question: What is the key finding from the study of LLM-based data augmentation for retrieval, concerning
the diminishing returns observed when scaling up the augmentation model? 

Synth Answer: Answer: The study finds that while augmentation improves retrieval models, its benefits do not scale 
indefinitely; beyond a certain point, additional augmentation leads to diminishing returns. This suggests that 
augmenting all documents is not necessary, and a smaller subset can be just as effective.

RAG Answer: So, you're looking for the key finding on diminishing returns. According to the study, it appears that 
there's a point of diminishing returns beyond a certain augmentation threshold. What that means is that while 
LLM-based data augmentation can indeed improve retrieval performance, its advantages don't scale indefinitely. 
augmentation becomes less effective beyond a certain scale, and augmenting the entire corpus is not always 
necessary. In fact, smaller augmentation models, like those with 8B parameters, can achieve performance comparable 
to larger models (70B) with less computational resources!

Synth Evaluation: [Score] 2
Justification: 
While both answers address the question of diminishing returns when scaling up the augmentation model, Answer 2 
does not introduce any inconsistencies. However, it provides additional insight and clarity by defining the 
augmentation threshold and highlighting the effectiveness of smaller models with fewer parameters and less 
computational resources. Answer 1 remains the ground truth and a clear concise answer to the question.

Set 2

Question: Question: What are the key findings and implications of incorporating retrieval information into 
retrieval augmented generation (R2AG)?

...

Synth Evaluation: [Score] 1 Justification

The second answer is somewhat inferior to the first answer and may introduce minor inconsistencies due to the 
following reasons:

* The second answer is mostly an excerpt from a research paper, which implies a level of bias and a specific 
perspective on the topic. The first answer, on the other hand, appears to be a more general summary of the key 
findings and implications of incorporating retrieval information into R2AG.
* The second answer provides specific results from an experimental paper, which may not be universally applicable 
or representative of other scenarios. The first answer, however, focuses on the general benefits and implications 
of R2AG, making it more applicable to a broader range of contexts.
* The second answer contains minor grammatical errors and awkward phrasing, which detract from its overall 
coherency and readability. The first answer, while not perfect, is well-written and easy to follow.
* The second answer introduces minor inconsistencies with the original question, as it focuses on the benefits of 
R2AG from a specific paper, whereas the first answer provides a more general overview of the topic.
 
Overall, while the second answer is generally correct and provides valuable insights, it lacks the comprehensive 
and coherent approach of the first answer.

Set 3

Question: Question: What are the key findings from the study that evaluated the effectiveness and scalability of 
LLM-based data augmentation for retrieval?

Synth Answer: Answer: The study found that while LLM augmentation is an effective strategy, its benefits do not 
scale indefinitely. The benefits of augmentation diminish beyond a certain threshold, and duplicating the entire 
dataset does not guarantee optimal performance. Additionally, the study demonstrated that smaller augmentation 
models, such as those with 8B parameters, can achieve performance comparable to larger models, and retrieving with 
strong pre-training shows diminishing gains from augmentation. Furthermore, the study found that using task-diverse
augmentations can mitigate performance drops observed in "Fact-Checking" tasks, but the saturation effects persist 
even with task-diverse augmentations. The study also showed that distillation from a larger model to a smaller 
model can bridge the performance gap, making fine-tuning on a small-scale curated subset of high-quality synthetic 
data a viable and cost-effective alternative to large-scale augmentation.

RAG Answer: The study you're referring to is "Evaluating the Effectiveness and Scalability of LLM-Based Data 
Augmentation for Retrieval." According to the findings, the analysis focused on the impact of augmentation scale 
and model size. The key takeaways are:

1. While augmentation can improve retrieval performance, its advantages do not scale indefinitely. There's a point 
of diminishing returns beyond a certain augmentation threshold, meaning you don't need to augment the entire 
corpus.
2. Smaller augmentation models, like those with 8B parameters, can achieve performance comparable to larger models 
(70B). This suggests a more computationally efficient alternative without sacrificing effectiveness.
3. Models with extensive pre-training show less improvement from augmentation, indicating that augmentation is 
particularly beneficial for models lacking robust pre-training.

These findings highlight the limitations and benefits of LLM-based data augmentation for retrieval, particularly in
relation to model size and pre-training. The study's insights can help inform strategies for optimizing retrieval 
performance in different scenarios.

Synth Evaluation: [Score] 0.8 Justification

This answer comes close to the ground truth, capturing the essential findings and insights from the study. The new 
answer correctly identifies the point of diminishing returns beyond a certain augmentation threshold (key takeaway 
1), the effectiveness of smaller augmentation models (key takeaway 2), and the less pronounced improvement from 
augmentation for models with extensive pre-training (key takeaway 3).

However, there are some inconsistencies and omissions compared to the original answer:

* Omission: The new answer does not mention the finding on task-diverse augmentations mitigating performance drops 
in "Fact-Checking" tasks.
* Inconsistency: The new answer states that the benefits of augmentation diminish beyond a certain threshold, 
whereas the original answer specifies that duplicating the entire dataset does not guarantee optimal performance. 
While both statements convey a similar idea, the original answer provides more nuanced context.
* Overemphasis: The new answer focuses on the positive aspects of augmentation, whereas the original answer also 
highlights the limitations and potential costs of large-scale augmentation.

Considering these factors, the score is 0.8. While the new answer is generally accurate and effectively 
communicates the key findings, it does not fully capture the complexity and richness of the original answer.

<br>

**Congratulations! We now have an LLM system that reasons about our pipeline and tries to evaluate it!** Now that we have some judge results, we can simply aggregate the results and see how often our formulation was according to an LLM:

In [7]:
# pref_score = sum(("[2]" in score) for score in pref_score) / len(pref_score)
# print(f"Preference Score: {pref_score}")

# Assume pref_scores_list holds evaluation strings
pref_scores_list = [
    "[2] The model performed well.",
    "[1] The model missed context.",
    "[2] Improved factual accuracy."
]

preference_score = sum(("[2]" in s) for s in pref_scores_list) / len(pref_scores_list)
print(f"Preference Score: {preference_score:.2f}")


Preference Score: 0.67


----

<br>

## **Part 4:** Advanced Formulations

The exercise above was meant to prepare you for the final assessment of the course and showcased a simple but effective evaluator chain. The objective and implementation details were provided for you, and the logic for using it probably makes sense now that you've seen it in action. 

With that being said, this metric was merely a product of us specifying:
- **What kind of behavior is important for our pipeline to have?**
- **What do we need to do in order to exhibit and evaluate this behavior?**

From these two questions, we could have come up with plenty of other evaluation metrics that could have assessed different attributes, incorporated different evaluator chain techniques, and even required different pipeline organization strategies. Though far from an exhaustive list, some common formulations you will likely come across may include:

- **Style Evaluation:** Some evaluation formulations can be as simple as "let me ask some questions and see if the output feels desirable." This might be used to see whether a chatbot "acts like it's supposed to" based on a description provided to a judge LLM. We're using quotations since this kind of assessment can reasonably be achieved with nothing but prompt engineering and a while loop.

- **Ground-Truth Evaluation:** In our chain, we used synthetic generation to create some random questions and answers using a sampling strategy, but in reality you may actually have some representative questions and answers that you need your chatbot to consistently get right! In this case, a modification of the exercise chain above should be implemented and closely monitored as you develop your pipelines.

- **Retrieval/Augmentation Evaluation:** This course made many assumptions about what kinds of preprocessing and prompting steps would be good for your pipelines, and much of this was determined by experimentation. Factors such as document preprocessing, chunking strategies, model selection, and prompt specification all played important roles, so creating metrics to validate these decisions may be of interest. This kind of metric might require your pipeline to output your context chunks or may even rely solely on embedding similarity comparisons, so keep this in mind when trying to implement a chain that works with multiple evaluation strategies. Consider the [**RagasEvaluatorChain**](https://docs.ragas.io/en/stable/howtos/integrations/langchain.html) abstraction as a decent starting point for making an custom generalizable evaluation routine. 

- **Trajectory Evaluation:** Using more advanced agent formulations, you can implement multiple-query strategies that assume the presence of conversational memory. With this, you can implement an evaluation agent which can:
    - Ask a series of questions in order to evaluate how well the agent is able to adapt and cater to the scenario. This kind of system generally considers a series of correspondence and aims to tease out and evaluate a "trajectory" of how the agent navigated the conversation. The [**LangChain Trajectory Evaluators documentation**](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/trajectory/) is a good starting point.
    - Alternatively, you could also implement an evaluation agent that tries to achieve objectives by interacting with the chatbot. Such an agent can output whether they were able to navigate to their solution in a natural manner, and can even be used to generate a report about the percieved performance. The [**LangChain Agents documentation**](https://python.langchain.com/v0.1/docs/modules/agents/) is a good starting point!

<br>

At the end of the day, just make sure to use the tools you have at your disposal appropriately. By this point in the course, you should already be well-acquainted with the LLM core value propositions: **They're powerful, scalable, predictable, controllable, and orchestratable... but will act unpredictably when you just expect them to work by default.** Assess your needs, formulate and validate your pipelines, give enough information, and add as much control as you can to make your system work consistently, efficiently, and effectively.

----

<br>

## **Part 5: [Assessment]** Evaluating For Credit

Welcome to the last exercise of the course! Hopefully you've enjoyed the material and are ready to actually get credit for these notebooks! For this part:

- **Make sure you're in the course environment**
- **Make sure `docstore_index/` has been uploaded to the course environment...**
    - **...and contains [at least one Arxiv paper](https://arxiv.org/search/advanced) which has been updated recently.**
- **Make sure you don't have some old session of [`09_langserve.ipynb`](09_langserve.ipynb) already occupying the port. Your assessment requires you to implement the new `/retriever` and `/generator` endpoints!!**

**Objective:** On launch, [**`frontend/frontend_block.py`**](frontend/frontend_block.py) had several lines of code which trigger the course pass condition. Your objective is to invoke that series of commands by using your pipeline to pass the **Evaluation** check! Recall [`09_langserve.ipynb`](09_langserve.ipynb) and use it as a starting example! As a recommendation, consider duplicating it so that you can keep the original as an authoritative reference. 

**Once Finished:** While your course environment is still open, please navigate back to your course environment launcher area and click the **"Assess Task"** button! After that, you're all done!

In [13]:
%%js
var url = 'http://'+window.location.host+':8090';
element.innerHTML = '<a style="color:green;" target="_blank" href='+url+'><h1>< Link To Gradio Frontend ></h1></a>';

<IPython.core.display.Javascript object>

----

<br>

## <font color="#76b900">**Congratulations On Completing The Course**</font>

Hopefully this course was not only exciting and challenging, but also adequately prepared you for work on the cutting edge of LLM and RAG system development! Going forward, you should have the skills necessary to tackle industry-level challenges and explore RAG deployment with open-source models and frameworks.

**Some NVIDIA-specific releases related to this that you may find interesting include:**
- [**NVIDIA NIM**](https://www.nvidia.com/en-us/ai/), which offers microservice spinup routines that can be deployed on local compute.
- [**TensorRT-LLM**](https://github.com/NVIDIA/TensorRT-LLM) is the current recommended framework for deploying GPU-accelerated LLM model engines in production settings.
- [**NVIDIA's Generative AI Examples Repo**](https://github.com/NVIDIA/GenerativeAIExamples), which includes the current canonical microservice example application and will be updated with new resources as new production workflows get released.
- [**The Knowledge-Based Chatbot Technical Brief**](https://resources.nvidia.com/en-us-generative-ai-chatbot-workflow/knowledge-base-chatbot-technical-brief) which discusses additional publicly-accessible details on productionalizing RAG systems.

**Additionally, some key topics you may be interested in delving more into include:**
- [**LlamaIndex**](https://www.llamaindex.ai/), which has strong components that can augment and occasionally improve upon the LangChain RAG features.
- [**LangSmith**](https://docs.smith.langchain.com/), an upcoming agent productionalization service offered by LangChain.
- [**Gradio**](https://www.gradio.app/), though touched on in the course, has many more interface options which will be worth investigating. For inspiration, consider checking out [**HuggingFace Spaces**](https://huggingface.co/spaces) for examples.
- [**LangGraph**](https://python.langchain.com/docs/langgraph/) is a framework for graph-based LLM orchestration, and is a natural next step forward for those interested in [multi-agent workflows](https://blog.langchain.dev/langgraph-multi-agent-workflows/).
- [**DSPy**](https://github.com/stanfordnlp/dspy), a flow engineering framework that allows you to optimize LLM orchestration pipelines based on empirical performance results.

<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>